# Late fusion — frozen SASRec + audio-content, validation-tuned weight

Joint embedding-fusion overfit (0.0581 < SASRec 0.0735). Here SASRec stays frozen; a training-free content score (user = mean audio of history, item = audio) is added with weight `beta` chosen on a **validation split of users**. `score = zscore(SASRec) + beta*zscore(content)`. beta=0 recovers SASRec, so fusion cannot do worse.

**Bar:** SASRec 0.0735. Accelerator → **GPU**, Internet On.

In [ ]:
!pip install -q --no-cache-dir --upgrade "git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
import os
os.environ["YMREC_DATA"] = "/kaggle/working/data"
import time, numpy as np, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
from ymrec.data.sequences import build_sequences
from ymrec.data.embeddings import load_aligned_embeddings

data = build_sequences(size="50m", maxlen=200)
content_emb, content_mask, dim = load_aligned_embeddings(data.item_ids)
print(f"items={data.n_items:,} eval={len(data.eval_pos):,} | content {content_emb.shape} coverage={content_mask.mean():.3f}")

In [ ]:
# 1) Train the SASRec backbone (the strong base).
from ymrec.models.sasrec import train_and_eval

t0 = time.time()
sasrec, best = train_and_eval(data, d=64, n_blocks=2, n_heads=1, dropout=0.2,
                              epochs=120, batch_size=128, lr=1e-3, eval_every=30)
print(f"SASRec trained in {(time.time()-t0)/60:.1f} min | best NDCG@10={best['ndcg@10']:.4f}")

In [ ]:
# 2) Tune the content fusion weight on a validation split, report on the held-out test split.
from ymrec.models.fusion import tune_fusion

res = tune_fusion(sasrec, content_emb, data, val_frac=0.5, seed=0)
print(f"\nbest beta = {res['best_beta']}")
print(f"test SASRec-only NDCG@10 = {res['test_sasrec_only']['ndcg@10']:.4f}")
print(f"test FUSED      NDCG@10 = {res['test_fused']['ndcg@10']:.4f}")

In [ ]:
import pandas as pd
f, s = res["test_fused"], res["test_sasrec_only"]
df = pd.DataFrame({
    "SASRec-only (test users)": {k: round(s[k], 4) for k in s},
    f"Fused beta={res['best_beta']} (test users)": {k: round(f[k], 4) for k in f},
})
df